# Train "Hey Omni" Wake Word

Custom OpenWakeWord model for Omni Vox servo-skull.
Based on the [automatic training notebook](https://github.com/dscripka/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb).

**Runtime:** Use GPU runtime (Runtime → Change runtime type → T4 GPU)

**Time:** ~45-60 minutes total

# 1. Environment Setup

In [ ]:
# Install dependencies (~3 min)

# piper-sample-generator for synthetic TTS training data
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
  'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# openWakeWord (full install for training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# Training dependencies
!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
  speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
  acoustics==0.2.6 tensorflow-cpu==2.8.1 tensorflow_probability==0.16.0 \
  onnx_tf==1.10.0 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19

# Download embedding models
import os
os.makedirs('./openwakeword/openwakeword/resources/models', exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx \
  -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite \
  -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx \
  -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite \
  -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

print('\n✅ Setup complete')

# 2. Download Training Data

In [ ]:
# Room impulse responses (~2 min)
import numpy as np
import scipy
import datasets
from tqdm import tqdm
from pathlib import Path

output_dir = './mit_rirs'
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print('✅ Room impulse responses downloaded')

In [ ]:
# Background noise audio (~5 min)

# AudioSet
os.makedirs('audioset', exist_ok=True)
fname = 'bal_train09.tar'
!wget -q -O audioset/{fname} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}
!cd audioset && tar -xf bal_train09.tar

output_dir = './audioset_16k'
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
audioset_dataset = audioset_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive (1 hour)
output_dir = './fma'
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
fma_dataset = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
for i in tqdm(range(120)):  # 1 hour of 30s clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

print('✅ Background noise downloaded')

In [ ]:
# Pre-computed negative features (~5 min download)
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print('✅ Feature data downloaded')

# 3. Configure Training

In [ ]:
import yaml

config = yaml.load(open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader)

# === HEY OMNI CONFIG ===
config['target_phrase'] = ['hey omni']
config['model_name'] = 'hey_omni'

# More samples = better model (5000 is a good balance of quality vs time)
config['n_samples'] = 5000
config['n_samples_val'] = 1000
config['steps'] = 25000  # More steps for better convergence

# Quality targets
config['target_accuracy'] = 0.7
config['target_recall'] = 0.5

# Data paths
config['background_paths'] = ['./audioset_16k', './fma']
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}

with open('hey_omni_config.yaml', 'w') as f:
    yaml.dump(config, f)

print(f'Target phrase: {config["target_phrase"]}')
print(f'Samples: {config["n_samples"]} positive, {config["n_samples_val"]} validation')
print(f'Training steps: {config["steps"]}')
print('✅ Config saved to hey_omni_config.yaml')

# 4. Train the Model

In [ ]:
import sys

# Step 1: Generate synthetic "hey omni" clips with Piper TTS (~10 min)
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_omni_config.yaml --generate_clips

In [ ]:
# Step 2: Augment clips with room acoustics + noise (~5 min)
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_omni_config.yaml --augment_clips

In [ ]:
# Step 3: Train the model (~15-20 min with T4 GPU)
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_omni_config.yaml --train_model

In [ ]:
# Step 4 (if tflite wasn't saved correctly): Manual conversion
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf
import tempfile

onnx_path = 'my_custom_model/hey_omni.onnx'
tflite_path = 'my_custom_model/hey_omni.tflite'

if os.path.exists(onnx_path):
    onnx_model = onnx.load(onnx_path)
    tf_rep = prepare(onnx_model, device='CPU')
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, 'tf_model'))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, 'tf_model'))
        tflite_model = converter.convert()
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)
    print(f'✅ Saved tflite model to {tflite_path}')
else:
    print('⚠️ ONNX model not found — check training output above')

# 5. Download Model

In [ ]:
# Download both model formats
from google.colab import files

for ext in ['onnx', 'tflite']:
    path = f'my_custom_model/hey_omni.{ext}'
    if os.path.exists(path):
        print(f'Downloading {path} ({os.path.getsize(path)} bytes)')
        files.download(path)
    else:
        print(f'⚠️ {path} not found')

print('\n🎉 Done! Upload hey_omni.tflite to your servo-skull.')
print('Usage: Model(wakeword_model_paths=["hey_omni.tflite"])')